In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import pandas as pd

eeg_path = "/content/drive/MyDrive/data_n_back_test/eeg/eeg.parquet"

In [ ]:
df = pd.read_parquet(eeg_path)

In [ ]:
# Filter phase 2
df = df[df["phase"] == 2].copy()

In [ ]:

# Filter phase 2
df = df[df["phase"] == 2].copy()

print(df.shape)
df.head()

(7694147, 142)


,timestamp,EEG.Counter,EEG.Interpolated,EEG.AF3,EEG.F7,EEG.F3,EEG.FC5,EEG.T7,EEG.P7,EEG.O1,...,POW.F8.Gamma,POW.AF4.Theta,POW.AF4.Alpha,POW.AF4.BetaL,POW.AF4.BetaH,POW.AF4.Gamma,datetime,subject,test,phase
82609,1.579252e+09,24,0,4206.922852,4209.871582,4217.051270,4212.307617,4206.666504,4210.000000,4215.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.000058,subject_01,1,2
82610,1.579252e+09,25,0,4201.794922,4201.153809,4211.410156,4210.769043,4207.948730,4208.846191,4216.666504,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.008038,subject_01,1,2
82611,1.579252e+09,26,0,4201.153809,4200.128418,4211.538574,4207.307617,4216.282227,4209.615234,4220.512695,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.015919,subject_01,1,2
82612,1.579252e+09,27,0,4199.102539,4196.538574,4208.333496,4201.538574,4203.461426,4210.128418,4212.692383,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.023899,subject_01,1,2
82613,1.579252e+09,28,0,4199.230957,4193.846191,4209.102539,4206.794922,4196.922852,4210.512695,4211.282227,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.031779,subject_01,1,2


In [ ]:
# Filter phase 2
df = df[df["phase"] == 2].copy()

In [ ]:
print(df.shape)
df.head()


(7694147, 142)


,timestamp,EEG.Counter,EEG.Interpolated,EEG.AF3,EEG.F7,EEG.F3,EEG.FC5,EEG.T7,EEG.P7,EEG.O1,...,POW.F8.Gamma,POW.AF4.Theta,POW.AF4.Alpha,POW.AF4.BetaL,POW.AF4.BetaH,POW.AF4.Gamma,datetime,subject,test,phase
82609,1.579252e+09,24,0,4206.922852,4209.871582,4217.051270,4212.307617,4206.666504,4210.000000,4215.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.000058,subject_01,1,2
82610,1.579252e+09,25,0,4201.794922,4201.153809,4211.410156,4210.769043,4207.948730,4208.846191,4216.666504,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.008038,subject_01,1,2
82611,1.579252e+09,26,0,4201.153809,4200.128418,4211.538574,4207.307617,4216.282227,4209.615234,4220.512695,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.015919,subject_01,1,2
82612,1.579252e+09,27,0,4199.102539,4196.538574,4208.333496,4201.538574,4203.461426,4210.128418,4212.692383,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.023899,subject_01,1,2
82613,1.579252e+09,28,0,4199.230957,4193.846191,4209.102539,4206.794922,4196.922852,4210.512695,4211.282227,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17 10:12:51.031779,subject_01,1,2


In [ ]:
df["test"].shape

(7694147,)

In [ ]:
df["test"] = df["test"] - 1

In [ ]:
print(df["test"].value_counts())
print("Subjects:", df["subject"].nunique())

test
2    2601877
1    2561659
0    2530611
Name: count, dtype: int64
Subjects: 16


In [ ]:
eeg_cols = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

X_raw = df[eeg_cols].values          # (N_total_samples, 14)

In [ ]:
df["subject"]

,subject
82609,subject_01
82610,subject_01
82611,subject_01
82612,subject_01
82613,subject_01
...,...
15218596,subject_16
15218597,subject_16
15218598,subject_16
15218599,subject_16


In [ ]:
FS = 128
WIN = 4 * FS      # 512
STRIDE = WIN      # no overlap

X_windows = []
y_windows = []
subjects = []

for subj, g in df.groupby("subject"):

    g = g.sort_index()   # ensure temporal order

    X = g[eeg_cols].values      # (samples, 14)
    y = g["test"].values        # workload labels

    n = len(X)

    for start in range(0, n - WIN + 1, STRIDE):
        end = start + WIN

        window = X[start:end]            # (512,14)
        label  = np.bincount(y[start:end]).argmax()

        X_windows.append(window.T)       # (14,512)
        y_windows.append(label)
        subjects.append(subj)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows, dtype=np.int64)
subjects  = np.array(subjects)

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("subjects:", subjects.shape)

X_windows: (15019, 14, 512)
y_windows: (15019,)
subjects: (15019,)


In [ ]:
time_ok = True

for subj, g in df.groupby("subject"):
    if not g["timestamp"].is_monotonic_increasing:
        print(f"{subj} is NOT time ordered")
        time_ok = False

if time_ok:
    print("All subjects are time ordered.")


All subjects are time ordered.


In [ ]:
for subj, g in df.groupby("subject"):
    diffs = np.diff(g["timestamp"].values)
    if np.any(diffs <= 0):
        print(f"{subj} has non-increasing timestamps")

subject_01 has non-increasing timestamps
subject_02 has non-increasing timestamps
subject_03 has non-increasing timestamps
subject_04 has non-increasing timestamps
subject_05 has non-increasing timestamps
subject_06 has non-increasing timestamps
subject_07 has non-increasing timestamps
subject_08 has non-increasing timestamps
subject_09 has non-increasing timestamps
subject_10 has non-increasing timestamps
subject_11 has non-increasing timestamps
subject_12 has non-increasing timestamps
subject_13 has non-increasing timestamps
subject_14 has non-increasing timestamps
subject_15 has non-increasing timestamps
subject_16 has non-increasing timestamps


In [ ]:
subj = "subject_01"
g = df[df["subject"] == subj]

ts = g["timestamp"].values

diffs = np.diff(ts)

print("Min diff:", diffs.min())
print("Any negative diffs?", np.any(diffs < 0))
print("Any zero diffs?", np.any(diffs == 0))

Min diff: 0.0
Any negative diffs? False
Any zero diffs? True


In [ ]:
import numpy as np

for subj, g in df.groupby("subject", sort=False):
    ts = g["timestamp"].to_numpy()
    d = np.diff(ts)

    has_decrease = np.any(d < 0)
    has_duplicates = np.any(d == 0)

    if has_decrease:
        print(subj, "has DECREASING timestamps (BAD)")
    elif has_duplicates:
        print(subj, "has DUPLICATE timestamps (OK if counter/sample index exists)")
    else:
        print(subj, "is strictly increasing (perfect)")

subject_01 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_02 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_03 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_04 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_05 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_06 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_07 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_08 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_09 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_10 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_11 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_12 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_13 has DUPLICATE timestamps (OK if counter/sample index exists)
subject_14 has DUPLICATE timestamps (OK if counter/sample index 

In [ ]:
subj = "subject_01"
g = df[df["subject"] == subj].sort_values(["timestamp", "EEG.Counter"])

ts = g["timestamp"].to_numpy()
cnt = g["EEG.Counter"].to_numpy()

print("timestamp decreases?", np.any(np.diff(ts) < 0))
print("counter decreases?", np.any(np.diff(cnt) < 0))
print("same timestamp count:", np.sum(np.diff(ts) == 0))
print("same ts but counter decreases?:",
      np.any((np.diff(ts) == 0) & (np.diff(cnt) < 0)))

timestamp decreases? False
counter decreases? True
same timestamp count: 487388
same ts but counter decreases?: False


In [ ]:
print("df rows:", len(df))
print("unique subjects:", df["subject"].nunique())
print("rows per subject (min/mean/max):",
      df.groupby("subject").size().min(),
      df.groupby("subject").size().mean(),
      df.groupby("subject").size().max())

df rows: 7694147
unique subjects: 16
rows per subject (min/mean/max): 463112 480884.1875 497565


In [ ]:
mixed = 0

for subj, g in df.groupby("subject"):
    g = g.sort_index()
    y = g["test"].values

    for start in range(0, len(y) - WIN + 1, STRIDE):
        segment = y[start:start+WIN]
        if len(np.unique(segment)) > 1:
            mixed += 1

print("Mixed-label windows:", mixed)

Mixed-label windows: 32


In [ ]:
X_windows = []
y_windows = []
subjects = []

for (subj, test), g in df.groupby(["subject", "test"]):

    g = g.sort_index()

    X = g[eeg_cols].values
    y = g["test"].values

    for start in range(0, len(X) - WIN + 1, STRIDE):
        end = start + WIN

        window = X[start:end]
        label  = y[start]   # no need for majority now

        X_windows.append(window.T)
        y_windows.append(label)
        subjects.append(subj)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows, dtype=np.int64)
subjects  = np.array(subjects)

print("X_windows:", X_windows.shape)

X_windows: (15007, 14, 512)


In [ ]:
mixed = 0

for (subj, test), g in df.groupby(["subject", "test"]):

    g = g.sort_index()
    y = g["test"].values

    for start in range(0, len(y) - WIN + 1, STRIDE):
        segment = y[start:start+WIN]
        if len(np.unique(segment)) > 1:
            mixed += 1

print("Mixed-label windows:", mixed)

Mixed-label windows: 0


In [ ]:
X_windows.shape

(15007, 14, 512)

In [ ]:
y_windows.shape

(15007,)

In [ ]:
# X_windows shape: (N, 14, 512)

mean = X_windows.mean(axis=2, keepdims=True)
std  = X_windows.std(axis=2, keepdims=True) + 1e-6

X_windows = (X_windows - mean) / std

print("After z-score:", X_windows.shape)
print("Mean (first window, first channel):", X_windows[0,0].mean())
print("Std  (first window, first channel):", X_windows[0,0].std())

After z-score: (15007, 14, 512)
Mean (first window, first channel): -7.778406e-06
Std  (first window, first channel): 1.0


In [ ]:

# -------------------------
# SETTINGS (edit these)
# -------------------------
WIN = 512
STRIDE = 256
FS = 128 # <-- set your dataset sampling rate

# Example bands (edit if you want)
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta",  12, 30),
]

print("df rows:", len(df))
print("unique subjects:", df["subject"].nunique())
print("unique tests:", df["test"].nunique())
print("eeg cols:", len(eeg_cols))

df rows: 7694147
unique subjects: 16
unique tests: 3
eeg cols: 14
